# K-Nearest Neighbours (K-NN) — from scratch

K-NN is a **lazy, instance-based** learner: "training" just stores the data. To predict a query point $\mathbf{x}$:

1. Compute the distance from $\mathbf{x}$ to every training point.
2. Take the $k$ closest points.
3. **Classification**: majority vote of their labels. **Regression**: average of their targets.

**Distance metrics** (Minkowski family):
$$d_p(\mathbf{a}, \mathbf{b}) = \Big(\sum_j |a_j - b_j|^p\Big)^{1/p} \quad p=1: \text{Manhattan},\; p=2: \text{Euclidean}$$

**Distance-weighted voting** gives closer neighbours more influence: weight $= 1/d$.

Because distances mix all features, **feature scaling matters**.

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

from collections import Counter

In [ ]:
def make_blobs(n_samples=300, centers=3, cluster_std=1.0, n_features=2,
               center_box=(-10, 10), random_state=None):
    """Isotropic Gaussian blobs (re-implementation of sklearn's make_blobs)."""
    rng = np.random.RandomState(random_state)
    if isinstance(centers, int):
        centers = rng.uniform(*center_box, size=(centers, n_features))
    centers = np.asarray(centers, dtype=float)
    k = len(centers)
    counts = [n_samples // k + (1 if i < n_samples % k else 0) for i in range(k)]
    stds = np.broadcast_to(np.asarray(cluster_std, dtype=float), (k,))
    X = np.vstack([rng.normal(c, s, size=(m, centers.shape[1]))
                   for c, s, m in zip(centers, stds, counts)])
    y = np.concatenate([np.full(m, i) for i, m in enumerate(counts)])
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def make_moons(n_samples=200, noise=0.1, random_state=None):
    """Two interleaving half circles."""
    rng = np.random.RandomState(random_state)
    n_out = n_samples // 2
    n_in = n_samples - n_out
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 0.5 - np.sin(t_in)]
    X = np.vstack([outer, inner]) + rng.normal(0, noise, size=(n_samples, 2))
    y = np.r_[np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)]
    idx = rng.permutation(n_samples)
    return X[idx], y[idx]

def train_test_split(X, y, test_size=0.25, random_state=None):
    """Shuffle and split arrays into train and test subsets."""
    rng = np.random.RandomState(random_state)
    idx = rng.permutation(len(X))
    n_test = int(round(len(X) * test_size))
    test, train = idx[:n_test], idx[n_test:]
    return X[train], X[test], y[train], y[test]

def accuracy_score(y_true, y_pred):
    return np.mean(np.asarray(y_true) == np.asarray(y_pred))

def plot_decision_boundary(model, X, y, title="", ax=None, resolution=150):
    """Colour the plane by the model's prediction and overlay the data."""
    ax = ax or plt.gca()
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, resolution),
                         np.linspace(y_min, y_max, resolution))
    Z = np.asarray(model.predict(np.c_[xx.ravel(), yy.ravel()]), dtype=float).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
    ax.set_title(title)

## Implementation

In [ ]:
def pairwise_distances(A, B, p=2):
    """Minkowski distance between every row of A and every row of B -> shape (len(A), len(B))."""
    if p == 2:   # fast path: ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a.b
        sq = (A ** 2).sum(1)[:, None] + (B ** 2).sum(1)[None, :] - 2 * A @ B.T
        return np.sqrt(np.maximum(sq, 0))
    return (np.abs(A[:, None, :] - B[None, :, :]) ** p).sum(-1) ** (1 / p)


class KNN:
    def __init__(self, k=5, p=2, weights="uniform", task="classification"):
        assert weights in ("uniform", "distance")
        self.k, self.p, self.weights, self.task = k, p, weights, task

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def _neighbours(self, X):
        D = pairwise_distances(np.asarray(X, dtype=float), self.X_train, self.p)
        idx = np.argsort(D, axis=1)[:, :self.k]
        return idx, np.take_along_axis(D, idx, axis=1)

    def predict(self, X, batch_size=2000):
        preds = []
        for start in range(0, len(X), batch_size):        # batching keeps memory bounded
            idx, dist = self._neighbours(X[start:start + batch_size])
            w = np.ones_like(dist) if self.weights == "uniform" else 1 / (dist + 1e-12)
            labels = self.y_train[idx]
            if self.task == "regression":
                preds.append((w * labels).sum(1) / w.sum(1))
                continue
            for row_labels, row_w in zip(labels, w):
                votes = Counter()
                for label, weight in zip(row_labels, row_w):
                    votes[label] += weight
                preds.append(votes.most_common(1)[0][0])
        return np.concatenate(preds) if self.task == "regression" else np.array(preds)

## Classification on 3 Gaussian blobs

In [ ]:
X, y = make_blobs(n_samples=450, centers=3, cluster_std=2.5, random_state=4)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

knn = KNN(k=5).fit(X_train, y_train)
print("Test accuracy (k=5):", accuracy_score(y_test, knn.predict(X_test)))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, k in zip(axes, [1, 15, 100]):
    m = KNN(k=k).fit(X_train, y_train)
    plot_decision_boundary(m, X_train, y_train, ax=ax,
                           title=f"k={k}, test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")
plt.tight_layout(); plt.show()

## Choosing $k$
Small $k$ → low bias, high variance (noisy boundary). Large $k$ → smooth but biased.
We pick $k$ by validation accuracy (odd $k$ avoids ties in binary problems).

In [ ]:
Xm, ym = make_moons(n_samples=600, noise=0.35, random_state=2)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.3, random_state=0)

ks = range(1, 60, 2)
results = {}
for weights in ["uniform", "distance"]:
    results[weights] = [accuracy_score(ym_test, KNN(k=k, weights=weights).fit(Xm_train, ym_train).predict(Xm_test))
                        for k in ks]
    plt.plot(ks, results[weights], "o-", ms=4, label=weights)
best_k = list(ks)[int(np.argmax(results["uniform"]))]
plt.xlabel("k"); plt.ylabel("test accuracy"); plt.legend(); plt.title(f"Accuracy vs k (best uniform k = {best_k})")
plt.show()

## Distance metrics

In [ ]:
for p, name in [(1, "Manhattan"), (2, "Euclidean"), (4, "Minkowski p=4")]:
    acc = accuracy_score(ym_test, KNN(k=best_k, p=p).fit(Xm_train, ym_train).predict(Xm_test))
    print(f"{name:14s}: {acc:.3f}")

## Why feature scaling matters
Add a noise feature with a huge scale: it dominates the distance and ruins K-NN until we standardise.

In [ ]:
rng = np.random.RandomState(0)
X_noisy_train = np.c_[Xm_train, rng.normal(0, 100, len(Xm_train))]
X_noisy_test = np.c_[Xm_test, rng.normal(0, 100, len(Xm_test))]
print("Unscaled with big noise feature:",
      accuracy_score(ym_test, KNN(k=best_k).fit(X_noisy_train, ym_train).predict(X_noisy_test)))

mu, sd = X_noisy_train.mean(0), X_noisy_train.std(0)
print("Standardised                    :",
      accuracy_score(ym_test, KNN(k=best_k).fit((X_noisy_train - mu) / sd, ym_train).predict((X_noisy_test - mu) / sd)))

## K-NN regression

In [ ]:
x = np.sort(np.random.uniform(0, 2 * np.pi, 120))
y_reg = np.sin(x) + np.random.normal(0, 0.2, len(x))
grid = np.linspace(0, 2 * np.pi, 400).reshape(-1, 1)

plt.scatter(x, y_reg, c="gray", s=12, label="data")
for k in [1, 10, 50]:
    plt.plot(grid, KNN(k=k, task="regression").fit(x.reshape(-1, 1), y_reg).predict(grid), label=f"k={k}")
plt.plot(grid, np.sin(grid), "k--", label="true sin(x)")
plt.legend(); plt.title("K-NN regression"); plt.show()

## Summary
* No training cost, but prediction is $O(n \cdot d)$ per query (KD-trees / ball trees speed this up).
* Sensitive to feature scale, irrelevant features, and the curse of dimensionality.
* $k$ trades off bias and variance; distance weighting often helps slightly.